# News Forecast Pipeline — 3 крупнейших СМИ РФ

**Цель:** Сбор → ETL → Анализ → Прогноз на **02.04.2026**

**СМИ:** Коммерсантъ, Лента.ру, Интерфакс

| Шаг | Ячейка | Описание |
|-----|--------|----------|
| 0 | Setup | Настройка окружения и импорты |
| 1 | Scrape | Сбор данных за 90 дней (RSS + архив) |
| 2 | ETL | Очистка, дедупликация, фильтрация |
| 3 | Analyze | Топики, частоты, NER, noise check |
| 4 | Forecast | Прогноз на целевую дату |
| 5 | Results | Просмотр итогового JSON |

---
## Ячейка 0 — Setup

In [ ]:
import sys, os, datetime
from dotenv import load_dotenv

# 1. Добавляем текущую директорию в путь импорта
sys.path.insert(0, os.getcwd())

# 2. Загружаем переменные окружения (.env)
load_dotenv()

import config, scraper, etl, analyzer, forecaster, backtester, metrics

print(f"Python: {sys.version.split()[0]}")
print(f"Рабочая директория: {os.getcwd()}")
print(f"Целевая дата прогноза: {config.TARGET_DATE}")

# Проверка наличия API ключа
if os.getenv('OPENROUTER_API_KEY'):
    print("✅ OpenRouter API Key: найден")
else:
    print("⚠️  OpenRouter API Key: НЕ НАЙДЕН (проверьте .env)")

---
## Ячейка 1 — Scrape: сбор данных

> ⏱️ **~5-30 минут** в зависимости от скорости соединения и количества доступных архивных страниц.
> 
> Можно ограничить список СМИ через `OUTLETS_TO_SCRAPE` или уменьшить `SCRAPE_FROM`.
> 
> Добавьте `enrich_leads=True` для полного скрапинга лидов (медленнее, по 1-2 сек/статья).

In [ ]:
import config, os, json
from scraper import scrape_all

# ── Настройки скрапинга ──────────────────────────────────────────
OUTLETS_TO_SCRAPE = config.OUTLET_SLUGS      # все 3, или например ["kommersant", "lenta"]
SCRAPE_FROM       = config.HISTORY_FROM       # дата начала (по умолчанию: TODAY − 90 дней)
SCRAPE_TO         = config.TODAY
ENRICH_LEADS      = False                    # True = доп. запросы за каждым лидом (медленно)
LOAD_EXISTING     = False                    # True = загрузить из JSON, если вы уверены, что он свежий
# ────────────────────────────────────────────────────────────────

if LOAD_EXISTING and os.path.exists(config.INTERMEDIATE_JSON):
    print(f"[pipeline] Загружаем существующий скрап из: {config.INTERMEDIATE_JSON}")
    with open(config.INTERMEDIATE_JSON, "r", encoding="utf-8") as f:
        scrape_results = json.load(f)
else:
    print("[pipeline] Файл не найден или LOAD_EXISTING=False. Запускаем скрапинг...")
    scrape_results = scrape_all(
        slugs=OUTLETS_TO_SCRAPE,
        start_date=SCRAPE_FROM,
        end_date=SCRAPE_TO,
        enrich_leads=ENRICH_LEADS,
    )

print("── Итог скрапинга ──")
for slug, recs in scrape_results.items():
    print(f"  {slug:<12} {len(recs):>5} записей")

---
## Ячейка 2 — ETL: очистка и дедупликация

In [ ]:
import importlib, etl as _etl_mod
importlib.reload(_etl_mod)
from etl import run_etl, load_clean
import pandas as pd

run_etl(slugs=config.OUTLET_SLUGS)

# Сводка по чистым данным
print("\n── Итог ETL ──")
summary_rows = []
for slug in config.OUTLET_SLUGS:
    df = load_clean(slug)
    if df.empty:
        summary_rows.append({"outlet": slug, "records": 0, "with_lead": 0,
                              "date_min": None, "date_max": None})
        continue
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    summary_rows.append({
        "outlet":   slug,
        "records":  len(df),
        "with_lead": df["lead"].notna().sum(),
        "date_min": df["published_at"].min().date() if not df.empty else None,
        "date_max": df["published_at"].max().date() if not df.empty else None,
    })

pd.DataFrame(summary_rows)


In [ ]:
# Просмотр нескольких строк для одного СМИ
PREVIEW_OUTLET = "kommersant"   # поменяйте на любой slug

df_preview = load_clean(PREVIEW_OUTLET)
df_preview["published_at"] = pd.to_datetime(df_preview["published_at"], errors="coerce")
df_preview.sort_values("published_at", ascending=False)[["published_at", "rubric", "title", "lead"]].head(10)

---
## Ячейка 3 — Analyze: темы, частоты, NER, noise check

In [ ]:
import config
from analyzer import analyze_all

analysis = analyze_all(slugs=config.OUTLET_SLUGS)


In [ ]:
# ── Топ-10 тем по каждому СМИ ────────────────────────────────────
import pandas as pd

TOPIC_OUTLET = "interfax"   # поменяйте на нужный slug

if TOPIC_OUTLET in analysis and analysis[TOPIC_OUTLET]:
    freq = analysis[TOPIC_OUTLET]["topic_freq"]
    print(f"\nТоп-10 тем ({config.OUTLETS[TOPIC_OUTLET]['name']}, последние {config.TOPIC_WINDOW} дней):\n")
    display(freq.head(10)[["cluster_name", "count", "pct"]])
else:
    print(f"Нет данных для {TOPIC_OUTLET} — сначала запустите Scrape+ETL")

In [ ]:
# ── Noise check: стабильность объёма по всем СМИ ─────────────────
noise_rows = []
for slug, res in analysis.items():
    if not res:
        continue
    n = res["noise"]
    noise_rows.append({
        "outlet":       slug,
        "daily_mean":   n.get("daily_mean"),
        "cv":           n.get("cv"),
        "stable":       "✅" if n.get("stable") else "⚠️",
        "top_rubric":   n.get("top_rubric"),
        "top_share_%":  round(n.get("top_rubric_share", 0) * 100, 1),
        "rubric_noisy": "⚠️" if n.get("rubric_noisy") else "✅",
    })

pd.DataFrame(noise_rows)

In [ ]:
# ── Топ сущностей (персоны, организации) ─────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ENT_OUTLET = "kommersant"   # поменяйте
ENT_TYPE   = "persons"  # persons | orgs | locations

if ENT_OUTLET in analysis and analysis[ENT_OUTLET]:
    ents = analysis[ENT_OUTLET]["entities"].get(ENT_TYPE, pd.Series())
    if not ents.empty:
        fig = go.Figure(go.Bar(
            x=ents.values[:20][::-1],
            y=ents.index[:20][::-1],
            orientation="h"
        ))
        fig.update_layout(
            title=f"Топ упоминаний: {ENT_TYPE} — {config.OUTLETS[ENT_OUTLET]['name']}",
            height=500, margin=dict(l=200)
        )
        fig.show()
    else:
        print(f"Нет сущностей типа '{ENT_TYPE}' для {ENT_OUTLET}")

In [ ]:
# ── Динамика публикаций по дням (все СМИ) ────────────────────────
import plotly.express as px
from etl import load_all_clean

df_all = load_all_clean()
if not df_all.empty:
    daily = (
        df_all.groupby([df_all["published_at"].dt.date, "outlet"])
              .size()
              .reset_index(name="count")
    )
    daily.columns = ["date", "outlet", "count"]
    fig = px.line(daily, x="date", y="count", color="outlet",
                  title="Количество публикаций по дням",
                  labels={"count": "Публикаций", "date": "Дата"})
    fig.show()
else:
    print("Нет данных — запустите Scrape + ETL")

---
## Ячейка 4 — Forecast: прогноз на 02.04.2026

In [ ]:
import config, datetime, os
from pathlib import Path
from forecaster import forecast_all

# ── Настройки ────────────────────────────────────────────────────
TARGET_DATE    = datetime.date(2026, 4, 2)
USE_LLM        = True   # False — пропустить генерацию заголовков через OpenRouter
FORECAST_SLUGS = config.OUTLET_SLUGS
FORECAST_STRATEGY = "llm"          # llm | hybrid
FORECAST_PROFILE  = "default"      # default | forward_look
# ────────────────────────────────────────────────────────────────

forecast_reports = forecast_all(
    slugs=FORECAST_SLUGS,
    target_date=TARGET_DATE,
    use_llm=USE_LLM,
    forecast_strategy=FORECAST_STRATEGY,
    forecast_profile=FORECAST_PROFILE,
)

forecast_pattern = f"forecast_{TARGET_DATE}_{FORECAST_STRATEGY}_{FORECAST_PROFILE}_*.json"
forecast_candidates = sorted(Path(config.FORECASTS_DIR).glob(forecast_pattern))
latest_forecast_json = str(forecast_candidates[-1]) if forecast_candidates else ""
latest_forecast_xlsx = latest_forecast_json.replace(".json", ".xlsx") if latest_forecast_json else ""

print(f"Strategy/Profile: {FORECAST_STRATEGY} / {FORECAST_PROFILE}")
if latest_forecast_json:
    print(f"JSON: {latest_forecast_json}")
if latest_forecast_xlsx and os.path.exists(latest_forecast_xlsx):
    print(f"XLSX: {latest_forecast_xlsx}")


In [ ]:
# ── Сводка по каждому СМИ ────────────────────────────────────────
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    name   = rep.get("outlet_name", slug)
    preds  = rep.get("predictions", [])
    llm    = [p for p in preds if p.get("method") == "llm"]

    print(f"\n{'='*60}")
    print(f"  {name} ({slug})")
    print(f"{'='*60}")
    print(f"  Режим: {rep.get('forecast_strategy', 'llm')} / {rep.get('forecast_profile', 'default')}")
    print(f"  Топ-темы: {', '.join(rep.get('top_topics', [])[:3])}")
    freq_topics = rep.get('frequency_top_topics', [])[:3]
    if freq_topics:
        print(f"  Частотные темы: {', '.join(freq_topics)}")
    print(f"  Контекст событий:\n    {rep.get('events_context', '').replace(chr(10), chr(10)+'    ')}")
    topic_selection = rep.get('topic_selection', [])[:3]
    if topic_selection:
        print("\n  Выбор тем:")
        for item in topic_selection:
            label = item.get('topic_label', '')[:70]
            ensemble = item.get('ensemble_score', 0.0)
            forward = item.get('forward_signal_score', 0.0)
            print(f"    · {label} | ensemble={ensemble:.2f} | forward={forward:.2f}")
    if llm:
        print(f"\n  LLM-заголовки ({len(llm)}):")
        for p in llm[:8]:
            print(f"    ▶ {p.get('title', '')}")
            lead = p.get("lead")
            if lead:
                print(f"      {lead[:120]}")
    else:
        print("\n  LLM-заголовки не сгенерированы.")

---
## Ячейка 5 — Results: просмотр итогового JSON

In [ ]:
import json, datetime
from pathlib import Path
from IPython.display import JSON

target_date = globals().get("TARGET_DATE", datetime.date(2026, 4, 2))
strategy = globals().get("FORECAST_STRATEGY", "llm")
profile = globals().get("FORECAST_PROFILE", "default")
pattern = f"forecast_{target_date}_{strategy}_{profile}_*.json"
matches = sorted(Path(config.FORECASTS_DIR).glob(pattern))

if matches:
    json_path = matches[-1]
    with open(json_path, encoding="utf-8") as f:
        forecast_json = json.load(f)
    print(f"Файл: {json_path}")
    print(f"СМИ в файле: {list(forecast_json.keys())}")
    # Интерактивный JSON-просмотр в Jupyter
    JSON(forecast_json)
else:
    print(f"Файл не найден по шаблону: {pattern}")
    print("Сначала запустите ячейку Forecast (шаг 6).")

In [ ]:
# ── Таблица всех LLM-прогнозов ────────────────────────────────────
import pandas as pd

rows = []
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    for p in rep.get("predictions", []):
        if p.get("method") == "llm":
            rows.append({
                "СМИ":     config.OUTLETS[slug]["name"],
                "Рубрика": p.get("rubric", ""),
                "Заголовок": p.get("title", ""),
                "Лид":     (p.get("lead") or "")[:120],
            })

if rows:
    df_llm = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", 80)
    display(df_llm)
else:
    print("LLM-прогнозы не сгенерированы. Проверьте OpenRouter или запустите с USE_LLM=True.")

In [ ]:
# ── Review-экспорт прогнозов в Excel ──────────────────────────────
import config, datetime, os
import pandas as pd

target_date = globals().get("TARGET_DATE", datetime.date(2026, 4, 2))
strategy = globals().get("FORECAST_STRATEGY", "llm")
profile = globals().get("FORECAST_PROFILE", "default")
EXPORT_PATH = os.path.join(config.FORECASTS_DIR, f"forecast_review_{target_date}_{strategy}_{profile}.xlsx")

rows = []
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    outlet_name = config.OUTLETS[slug]["name"]
    for p in rep.get("predictions", []):
        rows.append({
            "СМИ":         outlet_name,
            "Стратегия":   rep.get("forecast_strategy", strategy),
            "Профиль":     rep.get("forecast_profile", profile),
            "Метод":       p.get("method", ""),
            "Рубрика":     p.get("rubric", "") or p.get("topic_label", ""),
            "Заголовок":   p.get("title", "") or p.get("topic_label", ""),
            "Лид":         (p.get("lead") or ""),
            "Уверенность": p.get("score", ""),
        })

if rows:
    df_export = pd.DataFrame(rows)

    with pd.ExcelWriter(EXPORT_PATH, engine="openpyxl") as writer:
        # Лист 1: все прогнозы
        df_export.to_excel(writer, sheet_name="Все прогнозы", index=False)

        # Лист 2: только LLM-заголовки
        df_llm = df_export[df_export["Метод"] == "llm"]
        if not df_llm.empty:
            df_llm.to_excel(writer, sheet_name="LLM заголовки", index=False)

        # Лист 3+: по одному листу на СМИ
        for slug, rep in forecast_reports.items():
            if not rep:
                continue
            name = config.OUTLETS[slug]["name"]
            df_outlet = df_export[df_export["СМИ"] == name]
            df_outlet.to_excel(writer, sheet_name=slug[:31], index=False)

    print(f"✅ Файл сохранён: {EXPORT_PATH}")
    print("   Это вспомогательный review-экспорт; основной XLSX уже сохраняется автоматически в forecast_all.")
    print(f"   Строк: {len(df_export)}  |  Листов: {2 + len(forecast_reports)}")
else:
    print("⚠️  Нет данных для экспорта — сначала запустите ячейку Forecast.")
